<a href="https://colab.research.google.com/github/ahdrifai1234/data-science-2026/blob/main/Pertemuan10_Ahamd_Rifai_240401010260.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IDENTITAS

NAMA : Ahmad Rifai

NIM : 240401010260

KELAS : IF403

In [60]:
# Langkah 1: Muat dan Eksplorasi Data
import pandas as pd
# Membaca dataset langsung dari URL
df = pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")
print(df.shape)
print(df.head())
print(df["Churn"].value_counts(normalize=True))

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [61]:
# Langkah 2: Preprocessing
from sklearn.model_selection import train_test_split

# Drop customerID if it exists and is not needed for modeling
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

# Convert 'TotalCharges' to numeric, coercing errors to NaN and then filling with 0 or mean
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Separate X (fitur) dan y (target = Churn)
X = df.drop('Churn', axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0) # Convert 'Yes'/'No' to 1/0

# Encoding fitur kategorikal (mis. pd.get_dummies)
X = pd.get_dummies(X, drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(
X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Shape of X_tr: {X_tr.shape}")
print(f"Shape of X_te: {X_te.shape}")
print(f"Churn distribution in y_tr:\n{y_tr.value_counts(normalize=True)}")
print(f"Churn distribution in y_te:\n{y_te.value_counts(normalize=True)}")

Shape of X_tr: (5634, 30)
Shape of X_te: (1409, 30)
Churn distribution in y_tr:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn distribution in y_te:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [62]:
# Langkah 3: Latih Model
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [63]:
# Langkah 4: Evaluasi
from sklearn.metrics import classification_report, roc_auc_score

# hitung prediksi
y_pred = rf.predict(X_te)
# hitung probabilitas churn (untuk ROC-AUC)
y_proba = rf.predict_proba(X_te)[:, 1]

# tampilkan classification_report
print("Classification Report:")
print(classification_report(y_te, y_pred))

# tampilkan ROC-AUC
print(f"ROC-AUC Score: {roc_auc_score(y_te, y_proba)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8246208891988943


In [64]:
# Langkah 5: Prediksi Probabilitas dan Simpulkan
# hitung probabilitas churn (predict_proba)
churn_probabilities = rf.predict_proba(X_te)[:, 1]

print("Contoh probabilitas churn dari set pengujian:")
print(churn_probabilities[:10])

Contoh probabilitas churn dari set pengujian:
[0.         0.78666667 0.09       0.28       0.         0.41666667
 0.39333333 0.11       0.00666667 0.46      ]


### Kesimpulan
Model Random Forest telah dilatih dan dievaluasi untuk memprediksi churn pelanggan. Hasil evaluasi menunjukkan model memiliki kemampuan yang baik dalam membedakan pelanggan yang akan churn dan tidak, seperti yang ditunjukkan oleh skor ROC-AUC sebesar 0.82. Meskipun demikian, perlu dilakukan analisis lebih lanjut terhadap metrik presisi dan recall untuk kelas 'churn' untuk memastikan model efektif dalam mengidentifikasi pelanggan berisiko, dengan presisi 0.63 dan recall 0.50 untuk kelas churn.